<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/notebooks/stage_05_time_aware_data_splitting/stage_05_time_aware_data_splitting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **stage_05_time_aware_data_splitting**

## Introducción y Resumen

Esta notebook tiene como objetivo dividir el dataset MNQ en conjuntos de entrenamiento, validación y prueba, asegurando una partición aleatoria, reproducible y estructuralmente consistente. Con ello se dejan listos los datos para el entrenamiento y evaluación de los modelos predictivos.

0. Configuración del Entorno

    Se conecta Google Drive y se clona el repositorio de trabajo. Se instalan e importan librerías necesarias como pandas, numpy, matplotlib y seaborn. Se cargan los datasets procesados previamente (mnq_technical_indicators y mnq_alpha_factors) y se muestra un resumen de la información del dataset MNQ.

1. Carga de datos

    Se importa el dataset procesado con features técnicos y alpha factors. Se revisa su estructura (filas, columnas, tipos de datos) y también de importa el listado de features seleccionados para cada ventana de tiempo.

2. Análisis del dataset `mnq_model`

    Se revisa la estructura del dataset (filas, columnas, tipos de datos), se busca los valores NaNs y se verifica la distribución temporal de los registros.

3. Definición de parámetros de división

    En este punto se define la estrategia de partición del dataset: se toma un 70% de los días para entrenamiento, y el 30% restante se divide en partes iguales para validación y prueba. De esta manera, el modelo cuenta con suficientes datos para aprender, mientras que se reservan bloques temporales separados para ajustar parámetros y evaluar el rendimiento final sin fugas de información.

4. Selección aleatoria de días.

    Este punto busca garantizar que la partición de los datos sea representativa y no esté sesgada por la secuencia temporal. Al asignar los días de forma aleatoria —aunque de manera reproducible— se evita que los conjuntos queden condicionados por períodos específicos del mercado (por ejemplo, tendencias prolongadas o alta volatilidad en ciertos meses). Así, cada subconjunto refleja mejor la diversidad del dataset y se obtiene una evaluación más robusta del modelo.

5. Generación de datasets `mnq_train`, `mnq_test` y `mnq_valid`

    En este punto se crean los datasets mnq_train, mnq_valid y mnq_test, manteniendo homogeneidad en estructura (301 registros por día, de 09:30 a 14:30) y sin solapamiento entre conjuntos. Esto asegura consistencia en el entrenamiento, validación y prueba del modelo.

## 0. Configuración del Entorno


### 0.1. Clonado de repositorio / Acceso a Drive

In [ ]:
#Clonamos el repo
#LINK DE REPOSITORIO: https://github.com/GUNAPILLCO/neural_profit
#!git clone https://github.com/GUNAPILLCO/neural_profit.git

Cloning into 'neural_profit'...
remote: Enumerating objects: 530, done.
remote: Counting objects: 100% (150/150), done.
remote: Compressing objects: 100% (126/126), done.
remote: Total 530 (delta 94), reused 37 (delta 24), pack-reused 380 (from 2)
Receiving objects: 100% (530/530), 250.29 MiB | 15.65 MiB/s, done.
Resolving deltas: 100% (309/309), done.
Updating files: 100% (61/61), done.


In [2]:
from google.colab import drive
drive.mount('/content/drive')
drive_path = "/content/drive/MyDrive/neural_profit"

Mounted at /content/drive


### 0.2. Instalación de librerías


In [ ]:
#!{sys.executable} -m pip install -q ta
#print("Librería instalada: technical-analysis")

### 0.3. Importación de librerías


In [5]:
import sys
import re
#Instalación de librería pandas_market_calendars
#!{sys.executable} -m pip install -q pandas_market_calendars
#print("Librería instalada: pandas_market_calendars")


from functools import reduce
# Utilidades generales
from datetime import datetime, timedelta
import os
import glob
import requests
import warnings
warnings.filterwarnings('ignore')

# Manejo y procesamiento de datos
#import ta
import pandas as pd
import numpy as np
from tabulate import tabulate
import matplotlib.pyplot as plt
# Calendario de mercados
#import pandas_market_calendars as mcal

#from ta.momentum import StochasticOscillator, ROCIndicator
#from ta.volatility import BollingerBands, AverageTrueRange

from scipy.stats import spearmanr

import os
import json
import logging
from pathlib import Path
from typing import Dict, Any, List, Tuple

import numpy as np
import pandas as pd

#from ta.momentum import ROCIndicator

# ----------------------------
# Logging
# ----------------------------
logging.basicConfig(
    level=os.environ.get("LOG_LEVEL", "INFO"),
    format="%(asctime)s | %(levelname)s | %(message)s",
)
log = logging.getLogger("stage_05_time_aware_data_splitting")

In [9]:
# ============================================================
# Paths / IO (via env o defaults)
# ============================================================

DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))


IN_PARQUET = Path(os.environ.get("IN_PARQUET", "data/features/mnq_features_target.parquet"))

OUT_SPLITS = Path(os.environ.get("OUT_PARQUET", "data/splits/splits.json"))
OUT_PARQUET_TRAIN = Path(os.environ.get("OUT_PARQUET", "data/splits/mnq_train.parquet"))
OUT_PARQUET_VALID = Path(os.environ.get("OUT_PARQUET", "data/splits/mnq_valid.parquet"))
OUT_PARQUET_TEST = Path(os.environ.get("OUT_PARQUET", "data/splits/mnq_test.parquet"))

OUT_SUMMARY = Path(os.environ.get("OUT_SUMMARY", "reports/splits_summary.json"))


# Si en su pipeline hay un artifact de targets, puede quedar declarado,
# pero NO es estrictamente necesario en stage_04.
IN_ARTIFACT = Path(os.environ.get("IN_ARTIFACT", "reports/features_target_sumary.json"))

# PARA EL NOTEBOOK:
IN_PARQUET = DRIVE_DIR / IN_PARQUET
IN_ARTIFACT = DRIVE_DIR / IN_ARTIFACT

OUT_SPLITS = DRIVE_DIR / OUT_SPLITS
OUT_PARQUET_TRAIN = DRIVE_DIR / OUT_PARQUET_TRAIN
OUT_PARQUET_VALID = DRIVE_DIR / OUT_PARQUET_VALID
OUT_PARQUET_TEST = DRIVE_DIR / OUT_PARQUET_TEST
OUT_SUMMARY = DRIVE_DIR / OUT_SUMMARY


## 1. Carga de datos

### 1.1. Carga de dataset `mqn_to_model`




In [18]:
def load_data(data_path: str):

    #data_path = f'{drive_path}/2_feature_engineering/mnq_{data}.parquet'
    # Leer el archivo Parquet y cargarlo en un DataFrame
    df = pd.read_parquet(data_path)

    # Asegurar que el índice esté en formato datetime
    df.index = pd.to_datetime(df.index)

    # Crear una nueva columna 'date' con la fecha extraída del índice
    df['date'] = df.index.date

    # Reordenar columnas: 'date', 'time_str', y luego el resto
    cols = ['date'] + [col for col in df.columns if col not in ['date']]

    df = df[cols]

    return df

In [19]:
mnq_features_target = load_data(IN_PARQUET)

In [20]:
mnq_features_target

,date,open,high,low,close,price_ema60,momentum_10,roc_30,roc_60,delta_pts_60,delta_pts_90
1970-01-01 00:00:00.000000000,1970-01-01,8736.00,8736.75,8736.00,8736.75,0.000510,0.000143,0.100252,0.103119,0.25,0.50
1970-01-01 00:00:00.000000001,1970-01-01,8736.00,8736.00,8735.75,8735.75,0.000381,0.000143,0.100264,0.105999,0.75,2.25
1970-01-01 00:00:00.000000002,1970-01-01,8735.50,8735.50,8734.50,8735.00,0.000284,0.000086,0.080202,0.111745,1.00,3.00
1970-01-01 00:00:00.000000003,1970-01-01,8735.00,8735.75,8734.25,8734.50,0.000218,0.000086,0.071607,0.097410,2.25,3.25
1970-01-01 00:00:00.000000004,1970-01-01,8734.50,8734.50,8734.00,8734.00,0.000155,0.000029,0.060146,0.091680,3.75,3.50
...,...,...,...,...,...,...,...,...,...,...,...
1970-01-01 00:00:00.000548558,1970-01-01,21716.50,21722.75,21712.00,21719.50,-0.001801,-0.000207,-0.309818,-0.357839,-90.00,-102.00
1970-01-01 00:00:00.000548559,1970-01-01,21719.00,21719.75,21695.75,21698.50,-0.002675,-0.000714,-0.406206,-0.419917,-69.25,-74.75
1970-01-01 00:00:00.000548560,1970-01-01,21698.25,21700.25,21670.50,21679.25,-0.003444,-0.001577,-0.488852,-0.493419,-52.25,-57.50
1970-01-01 00:00:00.000548561,1970-01-01,21678.50,21684.25,21672.50,21681.50,-0.003231,-0.001807,-0.495652,-0.446536,-45.00,-53.50


In [7]:
def _ensure_parent_dir(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)


# ============================================================
# 1) Carga y preprocesamiento base
# ============================================================
def load_mnq_parquet(path: Path = IN_PARQUET) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el parquet de entrada: {path}")
    log.info(f"[OK] Cargando parquet: {path}")
    return pd.read_parquet(path)

def add_column_date(df: pd.DataFrame, date_col: str = "date") -> pd.DataFrame:
    """
    Asegura DatetimeIndex y agrega columna 'date' (YYYY-MM-DD) para agrupar por jornada.
    """
    out = df.copy()
    out.index = pd.to_datetime(out.index)
    out[date_col] = out.index.date
    # Reordenar (date primero)
    cols = [date_col] + [c for c in out.columns if c != date_col]
    return out[cols]


In [16]:
    #log.info("[1] Cargando mnq_intraday_labeled.parquet")
    mnq_features_target = load_mnq_parquet(IN_PARQUET)
    #mnq_features_target = add_column_date(mnq_features_target, date_col="date").sort_index()

In [17]:
mnq_features_target

,date,open,high,low,close,price_ema60,momentum_10,roc_30,roc_60,delta_pts_60,delta_pts_90
0,2019-12-23,8736.00,8736.75,8736.00,8736.75,0.000510,0.000143,0.100252,0.103119,0.25,0.50
1,2019-12-23,8736.00,8736.00,8735.75,8735.75,0.000381,0.000143,0.100264,0.105999,0.75,2.25
2,2019-12-23,8735.50,8735.50,8734.50,8735.00,0.000284,0.000086,0.080202,0.111745,1.00,3.00
3,2019-12-23,8735.00,8735.75,8734.25,8734.50,0.000218,0.000086,0.071607,0.097410,2.25,3.25
4,2019-12-23,8734.50,8734.50,8734.00,8734.00,0.000155,0.000029,0.060146,0.091680,3.75,3.50
...,...,...,...,...,...,...,...,...,...,...,...
548558,2025-06-13,21716.50,21722.75,21712.00,21719.50,-0.001801,-0.000207,-0.309818,-0.357839,-90.00,-102.00
548559,2025-06-13,21719.00,21719.75,21695.75,21698.50,-0.002675,-0.000714,-0.406206,-0.419917,-69.25,-74.75
548560,2025-06-13,21698.25,21700.25,21670.50,21679.25,-0.003444,-0.001577,-0.488852,-0.493419,-52.25,-57.50
548561,2025-06-13,21678.50,21684.25,21672.50,21681.50,-0.003231,-0.001807,-0.495652,-0.446536,-45.00,-53.50


### 1.2. Información de dataset MNQ_to_model


In [14]:
def info_dataset(df, name: str):
  print(f"Información del dataset {name}:\n")

  # Contar valores únicos en la columna 'date'
  num_dias = df['date'].nunique()
  print(f"\tCantidad de días: {num_dias}")

  # Filtrar valores válidos
  validos_por_dia = df.dropna(subset=['close']).groupby('date').size()

  # Calcular el promedio
  promedio_por_fecha = validos_por_dia.mean()
  print(f"\tRegistros por día: {int(promedio_por_fecha)}")

  primer_hora = df.index[0].strftime('%H:%M')
  ultima_hora = df.index[-1].strftime('%H:%M')
  zona_horaria = df.index[0].tzinfo


  print(f"\tHora diaria de inicio {primer_hora}")
  print(f"\tHora diaria de final {ultima_hora}")
  print(f"\tZona horaria: {zona_horaria}\n")

  return num_dias, promedio_por_fecha

In [15]:
num_dias, promedio_por_fecha = info_dataset(mnq_features_target, 'mnq_model')

Información del dataset mnq_model:

	Cantidad de días: 1
	Registros por día: 548563
	Hora diaria de inicio 00:00
	Hora diaria de final 00:00
	Zona horaria: None



### 1.3. Carga de listado de features por ventana de tiempo

In [ ]:
import json

# Ruta al archivo guardado
path = f'{drive_path}/2_feature_engineering/features_list.json'

with open(path, "r") as f:
    features_dict = json.load(f)

# Extraer las listas
features_to_30 = features_dict["features_to_30"]
features_to_60 = features_dict["features_to_60"]
features_to_90 = features_dict["features_to_90"]


In [ ]:
print(f'Listado de features para 30min: {features_to_30}')
print(f'Listado de features para 60min: {features_to_60}')
print(f'Listado de features para 90min: {features_to_90}')

Listado de features para 30min: ['ire_90', 'rev_mom_z_90', 'roc_60', 'rev_score_90', 'price_ema30']
Listado de features para 60min: ['ire_60', 'rev_mom_z_90', 'roc_60', 'bb_60', 'rev_mom_vol_z_60', 'momentum_5', 'roc_20']
Listado de features para 90min: ['ire_60', 'rev_mom_z_90', 'roc_60', 'bb_60', 'momentum_5', 'roc_20', 'rev_mom_vol_z_60']


## 2. Análisis del dataset `mnq_model`

### 2.0. Funciones

#### Función para contar NaN en dataset

In [ ]:
def nan_count(df):
  # Contar NaN por día y por columna
  daily_nan_counts = df.groupby("date").apply(lambda x: x.isna().sum())

  # Construir DataFrame con los valores únicos
  daily_unique_nans = pd.DataFrame({
      "feature": daily_nan_counts.columns,
      "daily_nan_counts": [sorted(daily_nan_counts[col].unique()) for col in daily_nan_counts.columns]
  })
  return daily_unique_nans

#### Función para detectar saltos temporales (gaps)

In [ ]:
def detectar_gaps(df: pd.DataFrame, gap_minutes: int = 1):
    """
    Verifica si existen saltos mayores al intervalo esperado (por defecto 1 minuto)
    entre registros consecutivos dentro de cada día, en un DataFrame con índice tipo DatetimeIndex.

    Omite el primer registro de cada día.

    Parámetros:
    - df: DataFrame con índice datetime.
    - gap_minutes: tamaño esperado del intervalo en minutos (por defecto 1).

    Retorna:
    - Lista de índices donde se detectaron diferencias mayores al intervalo esperado.
    """
    df = df.copy()
    df['time_diff'] = df.index.to_series().diff()

    base_time_diff = pd.Timedelta(minutes=gap_minutes)
    problem_indices = []

    for date, group in df.groupby(df.index.date):
        time_diff = group['time_diff'].iloc[1:]
        incorrect_indices = time_diff[time_diff != base_time_diff].index
        if len(incorrect_indices) > 0:
            problem_indices.append(incorrect_indices)

    if problem_indices:
        print(f"Se encontraron problemas en {len(problem_indices)} registros con diferencias irregulares.\n")

        # Conteo por fecha
        conteos = df.groupby(df.index.date).size()

        for i in range(len(problem_indices)):
            idx = problem_indices[i][0]
            diff = df.loc[idx, 'time_diff']
            date = idx.date()
            count = conteos[date]
            print(f'\t{idx} -> Diferencia: {diff} | # Registros: {count}')
    else:
        print("No se encontraron problemas, todas las muestras son consecutivas minuto a minuto.")

    return problem_indices

### 2.1. Búsqueda de NaN en `mnq_model`:

Buscamos los valores NaN en todas la columnas del dataset:

In [ ]:
mnq_model_nans = nan_count(mnq_model)
mnq_model_nans

,feature,daily_nan_counts
0,date,[0]
1,open,[0]
2,high,[0]
3,low,[0]
4,close,[0]
5,volume,[0]
6,target_return_30,[30]
7,target_return_60,[60]
8,target_return_90,[90]
9,bb_60,[59]


Lo que se observa es:

- OHLCV y date → [0] → nunca tienen valores faltantes.

- Targets (target_return_*) → [30], [60], [90] → todos los días tienen esos NaN al final, consistente con la ventana de predicción que corta datos futuros.

- Factores técnicos → muchos muestran valores únicos iguales al tamaño de la ventana usada en su cálculo:

  - bb_60 → [59] → se necesitan 60 valores para calcular, por eso hay 59 NaN iniciales cada día.
  - ire_60, roc_60, rev_mom_vol_z_60 → [60].
  - ire_90, rev_mom_z_90 → [90].
  - roc_20 → [20].
  - momentum_5 → [5].

Caso particular:

- rev_score_90 → [1, 2, 3] → parece que en algunos días puede generar hasta 3 NaN, pero no es fijo como los demás.

Las ventanas más grandes con valores NaN son la `ire_90` y `rev_mom_z_90` que necesitan 90 minutos de historial.

Vamos a filtrar el dataset `mnq_model` para eliminar todos los NaNs:

In [ ]:
# Filtrar filas sin NaN en ninguna columna
mnq_model_clean = mnq_model.dropna(how="any")

Verificamos si efectivamente no hay más NaNs en el dataset `mnq_model_clean`

In [ ]:
mnq_model_clean_nans = nan_count(mnq_model_clean)
mnq_model_clean_nans

,feature,daily_nan_counts
0,date,[0]
1,open,[0]
2,high,[0]
3,low,[0]
4,close,[0]
5,volume,[0]
6,target_return_30,[0]
7,target_return_60,[0]
8,target_return_90,[0]
9,bb_60,[0]


Se comprueba que no contamos con valores NaN. Ahora observemos la información del dataset:

In [ ]:
num_dias, promedio_por_fecha = info_dataset(mnq_model_clean, 'mnq_model_clean')

Información del dataset mnq_model_clean:

	Cantidad de días: 1311
	Registros por día: 301
	Hora diaria de inicio 09:30
	Hora diaria de final 14:30
	Zona horaria: America/New_York



Como se observa en el resultado, el primer registro válido (sin valores NaN en ninguna columna) aparece a las 09:30 y la jornada finaliza a las 14:30.

A continuación, verificamos en el dataset filtrado si existen saltos en la secuencia temporal o si todos los registros se mantienen consecutivos.

### 2.2. Búsqueda de gaps en `mnq_model_clean`

In [ ]:
detectar_gaps(mnq_model_clean)

No se encontraron problemas, todas las muestras son consecutivas minuto a minuto.


[]

Contamos con el dataset limpio de NaNs y saltos temporales.

## 3. Definición de parámetros de división

Para dividir el dataset en subconjuntos, se utiliza la siguiente estrategia:

- 70% de los días se asignan al conjunto de entrenamiento (train).

- El 30% restante se reparte de manera equitativa entre los conjuntos de validación (valid) y prueba (test).

Esto garantiza que el modelo disponga de la mayor parte de los datos para aprender patrones, mientras que las particiones de validación y prueba permiten ajustar hiperparámetros y evaluar el rendimiento fuera de muestra.

De esta forma, se asegura un esquema de división temporalmente consistente, sin solapamiento entre conjuntos.

In [ ]:
n_train = int(num_dias * 0.7)
print(f'Tamaño dataset train: {n_train} días')

n_valid = int((num_dias-n_train)/2)
print(f'Tamaño dataset valid: {n_val} días')

n_test = int((num_dias-n_train)/2)
print(f'Tamaño dataset test: {n_test} días')

Tamaño dataset train: 917 días
Tamaño dataset valid: 197 días
Tamaño dataset test: 197 días


## 4. Selección aleatoria de días.

En este punto nos aseguramos que la división de los datos no dependa únicamente del orden cronológico de los días, sino que se realice una selección aleatoria controlada. Para ello:

  - Se extraen los días únicos presentes en el dataset.
  - Se fija una semilla (`np.random.seed(42)`) para garantizar reproducibilidad en la mezcla.
  - Se aplica `np.random.shuffle` para reordenar los días de manera aleatoria.
  - Finalmente, se asignan los días a los conjuntos de entrenamiento, validación y prueba de acuerdo con los tamaños definidos previamente (917, 197 y 197 días respectivamente).

Este procedimiento nos permite que cada subconjunto mantenga independencia respecto a los demás, evitando sesgos por orden temporal y asegurando que los modelos se entrenen, validen y evalúen sobre muestras representativas del universo completo de días.

In [ ]:
# Obtener días únicos
unique_days = mnq_model['date'].unique()
np.random.seed(42)  # Reproducibilidad
np.random.shuffle(unique_days)  # Mezcla aleatoria

# Dividir días
train_days = unique_days[:n_train]
val_days = unique_days[n_train:n_train + n_val]
test_days = unique_days[n_train + n_val:n_train + n_val + n_test]

## 5. Generación de datasets `mnq_train`, `mnq_test` y `mnq_valid`

En este paso generamos los datasets finales para cada subconjunto: `mnq_train`, `mnq_valid` y `mnq_test`. La asignación se realiza filtrando los días correspondientes a cada conjunto, lo que asegura que no exista solapamiento entre ellos.

In [ ]:
# Crear datasets
mnq_train = mnq_model_clean[mnq_model_clean['date'].isin(train_days)].copy()
mnq_valid = mnq_model_clean[mnq_model_clean['date'].isin(val_days)].copy()
mnq_test = mnq_model_clean[mnq_model_clean['date'].isin(test_days)].copy()

Para un análisis posterior usamos la función antes definida `info_dataset` para confirmar que todos los subconjuntos comparten las mismas características estructurales:

In [ ]:
info_dataset(mnq_train, 'mnq_train')
info_dataset(mnq_valid,  'mnq_valid')
info_dataset(mnq_test, 'mnq_test')

Información del dataset mnq_train:

	Cantidad de días: 917
	Registros por día: 301
	Hora diaria de inicio 09:30
	Hora diaria de final 14:30
	Zona horaria: America/New_York

Información del dataset mnq_valid:

	Cantidad de días: 197
	Registros por día: 301
	Hora diaria de inicio 09:30
	Hora diaria de final 14:30
	Zona horaria: America/New_York

Información del dataset mnq_test:

	Cantidad de días: 197
	Registros por día: 301
	Hora diaria de inicio 09:30
	Hora diaria de final 14:30
	Zona horaria: America/New_York



(197, np.float64(301.0))

Se verifica que todos los datasets poseen una base homogénea, lo que nos va a facilitar la comparación de resultados entre etapas de entrenamiento, ajuste y evaluación. Además, la consistencia en el número de registros por día nos garantiza que los modelos reciban siempre ventanas de información con la misma extensión temporal.

## 6. Guardamos los datasets generados


In [ ]:
#Guardamos el dataset
ruta_mnq_train = f'{drive_path}/3_dataset_preparation/mnq_train.parquet'
mnq_train.to_parquet(ruta_mnq_train, index=True)

ruta_mnq_valid = f'{drive_path}/3_dataset_preparation/mnq_valid.parquet'
mnq_valid.to_parquet(ruta_mnq_valid, index=True)

ruta_mnq_test =  f'{drive_path}/3_dataset_preparation/mnq_test.parquet'
mnq_test.to_parquet(ruta_mnq_test, index=True)